$$
\huge \text{Recommendations System with Explainability}
$$

## Introduction

The goal of this project was to build a movie recommendation system for a streaming platform.

The idea was to create a system that can suggest movies matching user preferences. Since users may have different levels of activity on the platform, the project considered both users with existing rating history and users with no previous interactions.

To compare different recommendation strategies, three recommendation methods were implemented and evaluated. Additionally, MLflow was used to track experiments and compare model performance.


## 1. Dataset Overview - MovieLens 32M

The project uses the MovieLens 32M dataset.

The dataset contains:

* around 32 million ratings,
* around 2 million user-created tags,
* approximately 87 thousand movies,
* approximately 200 thousand users.

Ratings were used mainly for collaborative filtering methods, while movie information and tags supported content-based recommendations.

### Data Preparation

Before training the models, the data was prepared in several steps:

* loading and cleaning the data,
* splitting into training and testing sets,
* mapping user and movie IDs,
* creating input formats required by each recommendation method.

## 2. Recommendation Models

To compare different recommendation approaches, three recommendation methods were implemented.

### 2.1 Collaborative Filtering - Matrix Factorization

Collaborative filtering was implemented using two approaches:

* Singular Value Decomposition (SVD),
* Matrix Factorization trained with Stochastic Gradient Descent (SGD).

Both methods predict ratings by learning patterns from user–movie interactions.
Different parameter combinations were tested and tracked using MLflow.

Model performance was evaluated using:

* RMSE
* MAE

After comparing multiple runs, the best result was achieved by the SGD model with:

* k = 50
* epochs = 5
* learning rate = 0.005
* regularization = 0.01

Results:

* RMSE ≈ 0.819
* MAE ≈ 0.621

These results suggest that the SGD approach produced more accurate rating predictions than the tested SVD configurations.

### 2.2 Content-Based Filtering
Content-based filtering was implemented using movie metadata rather than user–user interactions. The model utilizes information about movie genres and user-generated tags to describe the content of each movie.

First, movie genres and tags were combined into a single textual representation. Genres separated by the "|" character were transformed into individual terms, while all tags associated with a movie were aggregated into one text field. The resulting descriptions were transformed into numerical feature vectors using the TF-IDF (Term Frequency–Inverse Document Frequency) technique.

Movie similarity was calculated using cosine similarity between TF-IDF vectors. Based on this representation, two recommendation strategies were implemented:
- Item-based content recommendations, where movies similar to a selected movie are recommended.
- User-based content recommendations, where a user profile is created from positively rated movies and recommendations are generated according to the similarity between the profile and available movies.

Model quality was evaluated using the Hit Rate@10 metric. For each user, the rating history was divided into training (80%) and test (20%) subsets. Recommendations were generated from the training data, and the model was considered successful if at least one movie from the test set appeared among the top 10 recommendations.

Results:
- Hit Rate@10 ≈ 0.532

The obtained results indicate that the model was able to recommend at least one relevant movie within the top 10 recommendations for approximately 53% of evaluated users. Considering the large size of the movie catalog, this demonstrates that genres and user-generated tags provide meaningful information for capturing user preferences and generating personalized recommendations.

## 3. Explainability

Explainability in machine learning is opening a "black box" and translating mathematics and algorithms to comprehensible language. Since Hybrid engine and Collaborative Filtering engine are based on complex matrix operations, we used surrogate model (Random Forest) in order to imitate the original engine. Then, we analyze this simplified model and measure its credibility with $R^2$ metrics.

Tools used:
- $\texttt{SHAP}$ library: we used this library to measure each feature's contribution to given model and final model's rating;
- $\texttt{RandomForestRegressor}$: the surrogate model imitating Hybrid engine and Collaborative Filtering engine, trained on recommendation sample in order to copy original engine's work;
- $\texttt{MinMaxScaler}$: we used this technique to normalize the ratings;
- $\texttt{r2\text{\textunderscore}score}$: we used this metrics to evaluate surrogate model's performance and its credibility (the higher $R^2$ score, the better the surrogate model performs);
- $\texttt{matplotlib}$: library used to generate plots.

### General conclusions

We know the user's preference profile thanks to the summary plot and beeswarm plot. We know which features globally matter the most for the user, i.e. we are able to deduce whether the user likes (red dots on positive semi-axis in beeswarm plot) or dislikes (red dots on negative semi-axis in beeswarm plot) a genre. What is more, we know why such recommendation was given to the user by looking at waterfall plot explaining which features raise or lower the final result. We know what features make the engine recommend or hide a specific movie to the user.

## 4. Cold Start Problem
### 4.1 Problem Description
One of the main challenges in recommender systems is the cold start problem. This issue occurs when a new user enters the system without any previous interaction history. Since recommendation algorithms typically rely on historical ratings or user behavior, generating personalized recommendations for a completely new user becomes difficult.

Therefore, an additional mechanism was implemented to collect initial preference information and create a user profile immediately after registration.

### 4.2 Proposed Solution
To address the cold start problem, an interactive onboarding procedure was developed.

The system presents a predefined list of popular and well-known movies from different genres, including animation, adventure, drama, action, historical, and science-fiction films. The user is asked to rate each movie on a scale from 0 to 5, where:
- 0 – not seen,
- 1 – strongly dislike,
- 2 – dislike,
- 3 – neutral,
- 4 – like,
- 5 – strongly like.

Half-point ratings are also supported.

After collecting the ratings, a new user profile is created and stored in the ratings dataset. Movies rated positively (rating ≥ 3.5) are used to construct an initial preference profile.

The collected information is then used in two ways:
- content-based recommendations are generated for each positively rated movie,
- a personalized user profile is created and passed to the user-based content recommendation model to generate final recommendations.

If the user does not provide any positive ratings, a fallback strategy is applied. In this case, a predefined set of popular movies is used to initialize the recommendation process and ensure that recommendations can still be generated.

### 4.3 Results

The implemented onboarding procedure successfully generated meaningful recommendations despite the absence of historical user data.

The content-based recommendations produced for individual movies were highly relevant. For example, positive ratings for Toy Story resulted in recommendations of similar animated movies such as Toy Story 2, Toy Story 3, Finding Nemo, Finding Dory, Monsters, Inc., and The Incredibles. Similarly, positive ratings for GoldenEye produced recommendations strongly related to the James Bond franchise.

The final personalized recommendations reflected multiple user interests simultaneously. Since the user provided positive ratings for movies representing different genres, including animation, adventure, historical drama, romance, and action, the generated profile became more diverse. Consequently, the recommendation system suggested movies from several thematic categories rather than focusing on a single genre.

These results indicate that the proposed solution effectively mitigates the cold start problem. By collecting a relatively small number of initial ratings, the system is able to generate personalized recommendations immediately after the first interaction with a new user.

### 4.4 Conclusion

The proposed cold start strategy successfully combines an interactive onboarding process with content-based recommendation techniques. The approach enables the system to generate personalized recommendations immediately after collecting a small number of ratings from a new user.